<a href="https://colab.research.google.com/github/shubhamgupta2702/AI-Agent-in-LangChain/blob/main/RAG_App_using_WeaviateDB_%26_HF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U langchain langchain-community langchain-core weaviate-client tiktoken pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.5/112.5 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 70.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 505.8/505.8 kB 54.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 619.5/619.5 kB 58.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 81.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 6.3 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.19
    Uninstalling langchain-core-1.2.19:
      Successfully uninstalled langchain-core-1.2.19
  Attempting uninst

In [19]:
pip install -U langchain-text-splitters

In [23]:
pip install langchain-huggingface

In [15]:
pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 31.3 MB/s eta 0:00:00


In [9]:
import weaviate
from weaviate.classes.init import Auth

In [2]:
from google.colab import userdata
weaviate_api_key = userdata.get('WEAVIATE_API_KEY')

In [69]:
import os
from google.colab import userdata

os.environ["HUGGINGFACEHUB_API_TOKEN"] = userdata.get("HUGGINGFACEHUB_API_TOKEN")

In [70]:
import os
print("Token exists:", os.environ.get("HUGGINGFACEHUB_API_TOKEN") is not None)

Token exists: True


In [4]:
WEAVIATE_URL="xfoi92phrxkgmtcxlbgoma.c0.asia-southeast1.gcp.weaviate.cloud"
WEAVIATE_API_KEY=weaviate_api_key

In [10]:
client = weaviate.connect_to_weaviate_cloud(
    cluster_url=WEAVIATE_URL,
    auth_credentials=Auth.api_key(weaviate_api_key),
)

print(client.is_ready())

True


In [13]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "/content/Transformers and LLMs.pdf"
loader = PyPDFLoader(file_path)

In [16]:
pages = loader.load()

In [21]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0)
texts = text_splitter.split_documents(pages)

In [24]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [27]:
collection = client.collections.create(
    name="Docs",
    vectorizer_config=None
)

with collection.batch.dynamic() as batch:
    for doc in texts:
        vector = embeddings.embed_query(doc.page_content)
        batch.add_object(
            properties={"text": doc.page_content},
            vector=vector
        )

In [34]:
query = "What is a vector database?"
query_vector = embeddings.embed_query(query)

In [35]:
results = collection.query.near_vector(
    near_vector=query_vector,
    limit=3,
    return_metadata=["distance"]  # optional but useful
)

In [37]:
for obj in results.objects:
    print("Text:", obj.properties["text"])
    print("Distance:", obj.metadata.distance)
    print("-" * 40)

Text: complex data learning 
capability.
Involves multiple layers of 
attention,  often used to manage 
different levels of granularity in 
the data.
Allowing the decoder to target 
specific parts of the encoder's  
output, essential for aligning  
input-output sequences.
Computes scores from query 
and key vectors, scaled by key 
dimensionality's square root, 
stabilizing gradients during 
training.
Uses extra information, like the 
decoder's  state in sequence-to- 
sequence models, to adjust its
Distance: 0.6181781888008118
----------------------------------------
Text: are replaced with transformer-based models:
• ViT Generator𝐺: This transformer-based generator𝐺 takes a latent vector𝑧 and produces an image or feature map. The
output of𝐺(𝑧)is typically processed by a series of transformer blocks that generate high-quality images or features.
• ViT Discriminator𝐷: This transformer-based discriminator𝐷 takes an image or feature map𝑥 and outputs a probability
that the input is from the

In [39]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
prompt = PromptTemplate(
    template="""You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use ten sentences maximum and keep the answer concise.
Question: {question}
Context: {context}
Answer:
""",
    input_variables=['context', 'question']
)

In [40]:
parser = StrOutputParser()

In [54]:
from typing import List
from langchain_core.documents import Document

class WeaviateV4Retriever:
    def __init__(self, collection, embedding_model, k=3):
        self.collection = collection
        self.embedding_model = embedding_model
        self.k = k

    def get_relevant_documents(self, query: str) -> List[Document]:
        query_vector = self.embedding_model.embed_query(query)

        results = self.collection.query.near_vector(
            near_vector=query_vector,
            limit=self.k,
            return_metadata=["distance"]
        )

        return [
            Document(
                page_content=obj.properties["text"],
                metadata={"distance": obj.metadata.distance}
            )
            for obj in results.objects
        ]

    # ✅ ADD THIS (makes it usable in chain)
    def __call__(self, query: str):
        return self.get_relevant_documents(query)

In [56]:
retriever = WeaviateV4Retriever(
    collection=collection,
    embedding_model=embeddings,
    k=3
)

docs = retriever.get_relevant_documents("What is a vector database?")

for d in docs:
    print(d.page_content, d.metadata)

complex data learning 
capability.
Involves multiple layers of 
attention,  often used to manage 
different levels of granularity in 
the data.
Allowing the decoder to target 
specific parts of the encoder's  
output, essential for aligning  
input-output sequences.
Computes scores from query 
and key vectors, scaled by key 
dimensionality's square root, 
stabilizing gradients during 
training.
Uses extra information, like the 
decoder's  state in sequence-to- 
sequence models, to adjust its {'distance': 0.6181781888008118}
are replaced with transformer-based models:
• ViT Generator𝐺: This transformer-based generator𝐺 takes a latent vector𝑧 and produces an image or feature map. The
output of𝐺(𝑧)is typically processed by a series of transformer blocks that generate high-quality images or features.
• ViT Discriminator𝐷: This transformer-based discriminator𝐷 takes an image or feature map𝑥 and outputs a probability
that the input is from the real data distribution. {'distance': 0.624150991

In [71]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
llm = HuggingFaceEndpoint(
  repo_id="Qwen/Qwen3-Coder-Next",
  task='text-generation'
)

model = ChatHuggingFace(llm=llm)

In [72]:
from langchain_core.runnables import RunnablePassthrough
chain = (
    {"context": retriever, "question":RunnablePassthrough()} | prompt | model | parser
)

In [73]:
chain.invoke("In which aspect the llms and transformers are different?")

'LLMs and Transformers differ primarily in scale and specialization. Transformers refer to the core architecture (e.g., encoder-decoder with self-attention). LLMs are large-scale models *built upon* this architecture. LLMs typically have billions of parameters, whereas early Transformers were smaller. LLMs often include fine-tuning, instruction-following, and multimodal capabilities. Transformers, as a framework, include models like BERT or vanilla seq2seq. LLM-specific challenges include data privacy and scaling inefficiencies. Transformers face general issues like attention bottlenecks. LLMs often undergo pretraining on massive corpora, then adapts via fine-tuning. Not all Transformers are LLMs, but all LLMs are Transformer-based.'